In [1]:
import torch
import wandb
import pandas as pd
from transformers import Trainer, Seq2SeqTrainer, Seq2SeqTrainingArguments

from Levenshtein import distance

import os
from datetime import datetime

os.chdir("../scripts")

from data_processing import poquad, processing
from t5.load_t5 import *

In [2]:
train_df, valid_df = poquad.load_poquad_manually_downloaded("../data/poquad-manually-processed")

In [3]:
train_input = poquad.dataset_into_str_input(train_df)
valid_input = poquad.dataset_into_str_input(valid_df)

In [26]:
models_to_evaluate = [
    # ("../models/plt5-small-8epochs", "plt5-small-8epochs"),
    ("plt5-original-small", "plt5-original-small"),
    ("plt5-original-base", "plt5-original-base"),
    ("../models/plt5-small-2epochs", "plt5-small-2epochs"),
    ("../scripts/results/checkpoint-22648", "plt5-small-2epochsV2"),
    ("../scripts/results/checkpoint-45296", "plt5-small-4epochs"),
    ("../scripts/results/checkpoint-67944", "plt5-small-6epochs"),
    # ("../scripts/results/checkpoint-90692", "plt5-small-8epochs")
 ]

In [23]:
# tokenizer, model = load_plt5("../scripts/results/checkpoint-22648")


In [27]:
for models_to_evaluate in models_to_evaluate:
    model_path, model_name = models_to_evaluate
    
    tokenizer, model = load_plt5(model_path)

    n_batches = len(valid_input) // 20
    gen_texts = []

    for i in range(n_batches):
        if i % 100 == 0:
            print(f"{model_name}:", i, "/", n_batches)

        sample = valid_input.iloc[i*20:(i+1)*20]

        tokenized_sample = tokenizer(sample["input_text"].to_list(), return_tensors="pt", padding=True, truncation=True, max_length=1024)

        generated_output = model.generate(**tokenized_sample, max_length=128, num_beams=4, num_return_sequences=1, no_repeat_ngram_size=2, early_stopping=True)

        gen_text = tokenizer.batch_decode(generated_output, skip_special_tokens=True)

        gen_texts.extend(gen_text)

    pd.Series(gen_texts, index=valid_df.index).to_json(f"../outputs/{model_name}_eval_0.json")

    with open("../outputs/plt5-small-8epochs_eval_0_gen_config.txt", "w") as f:
        f.writelines([
            "max_length=128", 
            "num_beams=4", 
            "num_return_sequences=1", 
            "no_repeat_ngram_size=2", 
            "early_stopping=True",
        ])

plt5-original-small: 0 / 353
plt5-original-small: 100 / 353
plt5-original-small: 200 / 353
